# 07 — Closures and Decorators

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
import functools

# Provided for this lesson.
from collections.abc import Callable

from lesson import Exercises, ___, run

check = Exercises('intermediate/07', {1: 'b66771111be58817', 2: 'e696f383856f7c97', 3: '1800795c889ca06b', 4: '8ccae967c818e7ac', 5: 'cd3ed912b9e1dc0e', 6: '822c00635404e1a3', 7: '38d21210e57ea8c5', 8: 'b7bd07f4077924bc', 9: '05e3468c158a617c', 10: 'd8b7818c71605aa7', 11: 'cb253b913f479017', 12: 'b66394cbe8bf74c9', 13: 'aa02835cc68ed0b1'}, practice='intermediate_02_iteration_and_abstractions', independent='once')

## 1. Functions are values

- A function can be stored in a variable, passed as an argument, and returned from another function.
- Pass the function itself (`len`), not its result (`len(x)`).
- `func.__name__` gives the function's name.

In [ ]:
def shout(text):
    return text.upper() + "!"

def apply(func, value):
    return func(value)

print(apply(shout, "hi"), apply(len, "hello"), shout.__name__)

**Exercise 1.** Pass `str.upper` to `apply`, without calling it yourself.

In [ ]:
def apply(func, value):
    return func(value)

result = apply(___, "abc")
check(1, result)

**Exercise 2.** Store each function in a dict and call the one for `"double"`.

In [ ]:
ops = {"double": lambda x: x * 2, "negate": lambda x: -x}
result = ops[___](7)
check(2, result)

**Exercise 3.** What does `total_price.__name__` give? Write the text.

In [ ]:
def total_price():
    pass

print(total_price.__name__)
predicted = ___
check(3, predicted)

## 2. Closures and nonlocal

- An inner function remembers the variables of the function it was defined in, even after that function returns. That is a closure.
- Reading an outer variable just works; to reassign it inside the inner function, declare it `nonlocal`.
- Each call to the outer function makes a new, independent closure.

In [ ]:
def make_counter():
    count = 0
    def next_value():
        nonlocal count
        count += 1
        return count
    return next_value

a, b = make_counter(), make_counter()
print(a(), a(), a(), b())

**Exercise 4.** Return a function that multiplies by `factor`.

In [ ]:
def make_multiplier(factor):
    def multiply(x):
        return x * ___
    return multiply

triple = make_multiplier(3)
check(4, triple(5))

**Exercise 5.** Make a tax adder: `make_tax_adder(0.2)(10)` adds 20% tax.

In [ ]:
def make_tax_adder(rate):
    def add_tax(price):
        return round(price * (1 + ___), 2)
    return add_tax

check(5, make_tax_adder(0.2)(10))

**Exercise 6.** Which keyword lets `next_value` reassign `count`? Write it.

In [ ]:
keyword = ___
check(6, keyword)

**Exercise 7.** Two counters from the demo, called in this order: `a(), a(), b()`. Predict the three results.

In [ ]:
predicted = ___
check(7, predicted)

## 3. Decorators

- A decorator takes a function and returns a new one that wraps it: `@trace` above `def f` means `f = trace(f)`.
- The wrapper accepts `*args, **kwargs`, does its extra work, calls the original, and returns its result.
- Use `@functools.wraps(func)` on the wrapper so the name and docstring are kept.

In [ ]:
import functools

def trace(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@trace
def add(a, b):
    return a + b

print(add(2, 3), add.__name__)

**Exercise 8.** Finish the wrapper so the decorated function still returns its result.

In [ ]:
def logged(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return ___
    return wrapper

@logged
def square(x):
    return x * x

check(8, square(6))

**Exercise 9.** Write a decorator that doubles whatever the function returns.

In [ ]:
def doubled(func):
    def wrapper(*args, **kwargs):
        return ___
    return wrapper

@doubled
def five():
    return 5

check(9, five())

**Exercise 10.** Without `functools.wraps`, what is `square.__name__` after decorating? Write the name.

In [ ]:
def plain(func):
    def wrapper(*args):
        return func(*args)
    return wrapper

@plain
def square(x):
    return x * x

name = ___
check(10, name)

## 4. Decorators with state and arguments

- A wrapper can keep state on itself: `wrapper.calls = 0`, then `wrapper.calls += 1` on each call.
- A decorator with arguments is a factory: `require_minimum(0)` returns the real decorator.
- That is three levels: factory(args) → decorator(func) → wrapper(*args).

In [ ]:
def require_minimum(minimum):
    def decorator(func):
        def wrapper(value, *args):
            if value < minimum:
                raise ValueError(f"{value} is below {minimum}")
            return func(value, *args)
        return wrapper
    return decorator

@require_minimum(0)
def withdraw(amount):
    return f"withdrew {amount}"

print(withdraw(5))
try:
    withdraw(-1)
except ValueError as error:
    print(error)

**Exercise 11.** Count calls on the wrapper.

In [ ]:
def count_calls(func):
    def wrapper(*args):
        wrapper.calls += 1
        return func(*args)
    wrapper.calls = ___
    return wrapper

@count_calls
def hello():
    return "hi"

hello(); hello(); hello()
check(11, hello.calls)

**Exercise 12.** Finish the factory: `@at_least(10)` rejects anything below 10.

In [ ]:
def at_least(minimum):
    def decorator(func):
        def wrapper(value):
            if value < minimum:
                return "rejected"
            return func(value)
        return wrapper
    return ___

@at_least(10)
def accept(value):
    return "ok"

check(12, (accept(3), accept(12)))

**Exercise 13.** Run the function only the first time; later calls return the remembered result.

In [ ]:
def once(func):
    results = []
    def wrapper(*args):
        if not results:
            results.append(func(*args))
        return results[___]
    return wrapper

@once
def setup(x):
    return x * 10

check(13, (setup(1), setup(2)))

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `make_tax_adder`

Return a function that adds `rate` tax, rounded to 2 decimals.

In [ ]:
def make_tax_adder(rate: float) -> Callable[[float], float]:
    """Return a function that adds `rate` tax to a price (rounded to 2 decimals)."""
    def add_tax(price: float) -> float:
        return ___
    return ___

In [ ]:
check('make_tax_adder')

### 2. `make_counter`

Return a function that gives 1, 2, 3, … on each call.

In [ ]:
def make_counter() -> Callable[[], int]:
    """Return a function that returns 1, 2, 3, ... on successive calls. Hint: nonlocal."""
    count = 0

    def counter() -> int:
        nonlocal count
        count += 1
        return ___
    return ___

In [ ]:
check('make_counter')

### 3. `trace`

Print `calling <name>` before each call; keep the result.

In [ ]:
def trace(func):
    """Decorator: print 'calling <name>' before each call and return the result unchanged.

    The wrapper must keep the original __name__ and __doc__ (functools.wraps).
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        return ___
    return ___

In [ ]:
check('trace')

### 4. `require_minimum`

A decorator factory that raises `ValueError` below a minimum.

In [ ]:
def require_minimum(minimum: float):
    """Decorator factory: raise ValueError if the first argument is below `minimum`."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(value, *args, **kwargs):
            if ___:
                raise ValueError(f"{func.__name__} needs a value >= {minimum}")
            return func(value, *args, **kwargs)
        return ___
    return ___

In [ ]:
check('require_minimum')

### Independent build — `once`

Wrap a callable so the first successful result is reused on later calls, even with different arguments. Forward positional and keyword arguments on the first call. Each decorated function has its own stored result.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def once(func):
    raise NotImplementedError

In [ ]:
check('once')

### 6. `count_calls`

Count calls in a `calls` attribute on the wrapper.

In [ ]:
def count_calls(func):
    """Decorator that counts calls in a `calls` attribute on the wrapper (starts at 0)."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return ___
    wrapper.calls = 0
    return ___

In [ ]:
check('count_calls')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** What does `nonlocal count` allow?

   - **a)** Assigning to a variable in the enclosing function
   - **b)** Reading a global
   - **c)** Creating a constant

**2.** Why use functools.wraps in a decorator?

   - **a)** Speed
   - **b)** It's required syntax
   - **c)** To keep the original __name__ and __doc__

**3.** @decorator above def f is the same as…

   - **a)** f = decorator(f)
   - **b)** decorator = f(decorator)
   - **c)** f(decorator)

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/07', {1: '4892c802d94b', 2: '1b7a71ef164f', 3: 'b10d41e834f9'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [08 — Context Managers and Resource Safety](08_context_managers_and_resources.ipynb)